
# QML-SleepNet — FINAL Fixed Equal-Logit Ensemble Freeze v1

## Purpose

Close Task A with **one fixed zero-training ensemble** made from three already-computed, architecturally distinct branches:

1. `bridge` — classical 128→64→32→8 bridge classifier;
2. `qt_angle_rx` — Stage04 Angle-Rx Quantum Transformer classifier;
3. `current_stage06` — corrected Stage06 ECG-temporal + causal + VQC hybrid.

Each branch is processed independently:

**raw probability → one NLL temperature → prior-corrected HMM (λ=1) → forward-backward Apnea posterior**

The three HMM posteriors are then combined with **fixed equal weights in log-odds space**:

\[
p_{\mathrm{ens}} =
\sigma\left(
\frac{
\operatorname{logit}(p_B)
+\operatorname{logit}(p_{QT})
+\operatorname{logit}(p_{S6})
}{3}
\right)
\]

Hard decision remains fixed at **0.5**.

---

## Hard lock

This notebook performs:

- NO neural retraining;
- NO quantum retraining;
- NO quantum simulation;
- NO new feature extraction;
- NO candidate search;
- NO ensemble-weight search;
- NO pairwise/subset tournament;
- NO threshold search;
- NO HMM λ search;
- NO smoothing/window search;
- NO official-x label access.

The exact ensemble is already fixed:

`bridge + qt_angle_rx + current_stage06`

with weights:

`[1/3, 1/3, 1/3]`

and blend family:

`equal logit mean`.

---

## Development gate

Before x prediction freeze, the fixed ensemble must beat the strict Bridge development winner by:

1. accuracy gain ≥ **+0.25 percentage points**;
2. balanced accuracy non-decrease;
3. F1 non-decrease;
4. AUPRC non-decrease;
5. sensitivity loss no worse than **−2.0 percentage points**.

If any gate fails, **STOP and retain Strict Bridge**.

---

## Scientific chronology

Official x labels had already been observed historically before this exact fixed ensemble was proposed.

Therefore any later x score is:

**post-hoc extended-pipeline official x-set evidence**

—not a pristine untouched external-test estimate.


In [1]:

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from datetime import datetime, timezone
import hashlib, json
import numpy as np
import pandas as pd
import torch

from scipy.optimize import minimize_scalar

from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, matthews_corrcoef, roc_auc_score, average_precision_score,
    confusion_matrix
)

ROOT = Path("/content/drive/MyDrive/QML_SleepNet")

STRICT = (
    ROOT / "outputs/GUIDE_EXACT_METRICMAX"
    / "EXISTING_ARTIFACT_STRICT_OUTER_FUSION_CORRECTION_v1"
)
STRICT_OOF = STRICT / "STRICT_OUTER_OOF_SCORES.npz"
STRICT_DECISION = STRICT / "STRICT_OUTER_FINAL_DECISION.json"

BRIDGE = ROOT / "outputs/GUIDE_EXACT_METRICMAX/04_bridge128to8_v1_3_fullgrid"
BRIDGE_PT = BRIDGE / "final_fit/bridge128to8_model.pt"
BRIDGE_QREADY = BRIDGE / "final_fit/quantum_ready_8d.npz"

STAGE04 = ROOT / "outputs/GUIDE_EXACT_METRICMAX/04_quantum_module_v3_audited"
QT_PT = STAGE04 / "final_fit/final_qt_angle_rx.pt"
QFEATURES = STAGE04 / "final_fit/quantum_features_for_final_stage06.npz"

STAGE06 = (
    ROOT / "outputs/GUIDE_EXACT_METRICMAX"
    / "05B_06_FINAL_GUIDE_CORRECTED_CONSOLIDATED_v1"
)
STAGE06_FINAL = STAGE06 / "final_fit/final_predictions.npz"

OUT = (
    ROOT / "outputs/GUIDE_EXACT_METRICMAX"
    / "FINAL_FIXED_EQUAL_LOGIT_ENSEMBLE_v1"
)
OUT.mkdir(parents=True, exist_ok=True)

EPS = 1e-8
LAPLACE = 1.0
HMM_LAMBDA = 1.0
HARD_THRESHOLD = 0.5

ENSEMBLE_BRANCHES = ["bridge", "qt_angle_rx", "current_stage06"]
ENSEMBLE_WEIGHTS = np.asarray([1/3, 1/3, 1/3], dtype=np.float64)

for p in [
    STRICT_OOF, STRICT_DECISION,
    BRIDGE_PT, BRIDGE_QREADY,
    QT_PT, QFEATURES,
    STAGE06_FINAL,
]:
    if not p.is_file():
        raise FileNotFoundError(p)

print("FINAL FIXED ENSEMBLE")
print("Branches:", ENSEMBLE_BRANCHES)
print("Weights:", ENSEMBLE_WEIGHTS.tolist())
print("Blend: equal mean in logit space")
print("Training: NO")
print("Quantum simulation: NO")
print("Official-x labels: NOT LOADED")


Mounted at /content/drive
FINAL FIXED ENSEMBLE
Branches: ['bridge', 'qt_angle_rx', 'current_stage06']
Weights: [0.3333333333333333, 0.3333333333333333, 0.3333333333333333]
Blend: equal mean in logit space
Training: NO
Quantum simulation: NO
Official-x labels: NOT LOADED


In [2]:

# Helpers

def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def atomic_json(path, obj):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, indent=2, default=str))
    tmp.replace(path)

def atomic_csv(path, df):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    df.to_csv(tmp, index=False)
    tmp.replace(path)

def metrics(y_true, score, threshold=0.5):
    y_true = np.asarray(y_true, dtype=np.int8)
    score = np.asarray(score, dtype=np.float64)
    pred = (score >= threshold).astype(np.int8)

    tn, fp, fn, tp = confusion_matrix(
        y_true, pred, labels=[0, 1]
    ).ravel()

    return {
        "n": int(len(y_true)),
        "accuracy": float(accuracy_score(y_true, pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, pred)),
        "precision": float(precision_score(y_true, pred, zero_division=0)),
        "sensitivity": float(recall_score(y_true, pred, zero_division=0)),
        "specificity": float(tn / max(tn + fp, 1)),
        "f1": float(f1_score(y_true, pred, zero_division=0)),
        "mcc": float(matthews_corrcoef(y_true, pred)),
        "auroc": float(roc_auc_score(y_true, score)),
        "auprc": float(average_precision_score(y_true, score)),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }

def logit(p):
    p = np.clip(np.asarray(p, dtype=np.float64), EPS, 1-EPS)
    return np.log(p) - np.log1p(-p)

def sigmoid(z):
    z = np.asarray(z, dtype=np.float64)
    out = np.empty_like(z)
    pos = z >= 0
    out[pos] = 1.0 / (1.0 + np.exp(-z[pos]))
    ez = np.exp(z[~pos])
    out[~pos] = ez / (1.0 + ez)
    return out

def equal_logit_mean(*scores):
    S = np.column_stack([
        np.asarray(s, dtype=np.float64) for s in scores
    ])
    if S.shape[1] != 3:
        raise RuntimeError("Fixed ensemble requires exactly 3 branches")
    return sigmoid(np.mean(logit(S), axis=1))

def fit_temperature(prob, y):
    p = np.clip(np.asarray(prob, float), EPS, 1-EPS)
    yy = np.asarray(y, float)
    lg = np.log(p) - np.log1p(-p)

    def objective(logT):
        z = lg / np.exp(logT)
        return float(np.mean(np.logaddexp(0.0, z) - yy*z))

    r = minimize_scalar(
        objective,
        method="bounded",
        bounds=(-4.0, 4.0),
        options={"xatol":1e-8, "maxiter":500},
    )

    if not r.success:
        raise RuntimeError(r.message)

    return float(np.exp(r.x)), float(r.fun)

def temperature_scale(prob, T):
    p = np.clip(np.asarray(prob, float), EPS, 1-EPS)
    z = (np.log(p) - np.log1p(-p)) / float(T)

    out = np.empty_like(z)
    pos = z >= 0
    out[pos] = 1.0 / (1.0 + np.exp(-z[pos]))

    ez = np.exp(z[~pos])
    out[~pos] = ez / (1.0 + ez)

    return np.clip(out, EPS, 1-EPS)

def parse_uid(uids):
    uids = np.asarray(uids).astype(str)
    rec = np.asarray([u.rsplit(":",1)[0] for u in uids])
    ep = np.asarray([int(u.rsplit(":",1)[1]) for u in uids], dtype=np.int64)
    return rec, ep

def contiguous_segments(indices, uids):
    indices = np.asarray(indices, dtype=np.int64)
    rec, ep = parse_uid(uids)

    out = []

    for r in np.unique(rec[indices]):
        rr = indices[rec[indices] == r]
        order = rr[np.argsort(ep[rr])]
        ee = ep[order]

        cuts = [0] + (np.where(np.diff(ee) != 1)[0] + 1).tolist() + [len(order)]

        out.extend(
            order[a:b]
            for a,b in zip(cuts[:-1], cuts[1:])
            if b > a
        )

    return out

def estimate_hmm(uids, y):
    y = np.asarray(y, dtype=np.int8)
    idx = np.arange(len(y), dtype=np.int64)

    counts = np.bincount(y, minlength=2).astype(np.float64)
    prior = counts / counts.sum()

    init = np.full(2, LAPLACE, dtype=np.float64)
    trans = np.full((2,2), LAPLACE, dtype=np.float64)

    for seg in contiguous_segments(idx, uids):
        sy = y[seg]
        init[int(sy[0])] += 1.0

        for a,b in zip(sy[:-1], sy[1:]):
            trans[int(a), int(b)] += 1.0

    pi = init / init.sum()
    A = trans / trans.sum(axis=1, keepdims=True)

    return prior, pi, A

def logsumexp(v):
    v = np.asarray(v, dtype=np.float64)
    m = np.max(v)
    return float(m + np.log(np.exp(v-m).sum()))

def forward_backward(prob, prior, pi, A):
    p = np.clip(np.asarray(prob, float), EPS, 1-EPS)
    post = np.column_stack([1-p, p])

    logE = (
        np.log(post)
        - HMM_LAMBDA * np.log(np.clip(prior, EPS, 1))[None,:]
    )
    logE -= np.max(logE, axis=1, keepdims=True)

    Tn = len(p)
    lpi = np.log(np.clip(pi, EPS, 1))
    lA = np.log(np.clip(A, EPS, 1))

    alpha = np.full((Tn,2), -np.inf)
    beta = np.full((Tn,2), -np.inf)

    alpha[0] = lpi + logE[0]

    for t in range(1,Tn):
        for s in range(2):
            alpha[t,s] = logE[t,s] + logsumexp(alpha[t-1] + lA[:,s])

    ll = logsumexp(alpha[-1])
    beta[-1] = 0.0

    for t in range(Tn-2,-1,-1):
        for s in range(2):
            beta[t,s] = logsumexp(lA[s] + logE[t+1] + beta[t+1])

    gamma = np.exp(alpha + beta - ll)
    gamma /= gamma.sum(axis=1, keepdims=True)

    return gamma[:,1]

def decode_external(raw_prob, uids, T, prior, pi, A):
    pcal = temperature_scale(raw_prob, T)
    idx = np.arange(len(uids), dtype=np.int64)
    out = np.full(len(uids), np.nan, dtype=np.float64)

    for seg in contiguous_segments(idx, uids):
        out[seg] = forward_backward(
            pcal[seg], prior, pi, A
        )

    if not np.isfinite(out).all():
        raise RuntimeError("HMM decode incomplete")

    return out

def torch_load(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")

def linear_head_probability(checkpoint_path, features):
    ck = torch_load(checkpoint_path)
    sd = ck["state_dict"]

    W = sd["head.weight"].detach().cpu().numpy().astype(np.float64)
    b = sd["head.bias"].detach().cpu().numpy().astype(np.float64)

    X = np.asarray(features, dtype=np.float64)

    logits = X @ W.T + b[None,:]
    logits -= logits.max(axis=1, keepdims=True)

    e = np.exp(logits)
    return e[:,1] / e.sum(axis=1)

print("Helpers ready.")


Helpers ready.


In [3]:

# ------------------------------------------------------------
# DEVELOPMENT REPRODUCTION — ONE FIXED RULE, NO SEARCH
# ------------------------------------------------------------

decision = json.loads(STRICT_DECISION.read_text())

if decision.get("official_x_labels_used") is not False:
    raise RuntimeError("Strict development provenance failure")

if decision.get("development_winner") != "bridge":
    raise RuntimeError(
        f"Expected strict development winner bridge, got {decision.get('development_winner')}"
    )

z = np.load(STRICT_OOF, allow_pickle=False)

UID_DEV = np.asarray(z["uids"]).astype(str)
Y_DEV = np.asarray(z["y"], dtype=np.int8)

P_BRIDGE_DEV = np.asarray(z["bridge"], dtype=np.float64)
P_QT_DEV = np.asarray(z["qt_angle_rx"], dtype=np.float64)
P_S6_DEV = np.asarray(z["current_stage06"], dtype=np.float64)

P_FIXED_DEV = equal_logit_mean(
    P_BRIDGE_DEV,
    P_QT_DEV,
    P_S6_DEV,
)

BRIDGE_DEV = metrics(Y_DEV, P_BRIDGE_DEV, HARD_THRESHOLD)
FIXED_DEV = metrics(Y_DEV, P_FIXED_DEV, HARD_THRESHOLD)

delta = {
    k: float(FIXED_DEV[k] - BRIDGE_DEV[k])
    for k in [
        "accuracy","balanced_accuracy","precision","sensitivity",
        "specificity","f1","mcc","auroc","auprc"
    ]
}

GATE = {
    "accuracy_gain_pp": 100.0 * delta["accuracy"],
    "minimum_accuracy_gain_pp": 0.25,
    "balanced_accuracy_nondecrease": delta["balanced_accuracy"] >= 0.0,
    "f1_nondecrease": delta["f1"] >= 0.0,
    "auprc_nondecrease": delta["auprc"] >= 0.0,
    "sensitivity_loss_no_worse_than_2pp": delta["sensitivity"] >= -0.020,
}
GATE["pass"] = bool(
    GATE["accuracy_gain_pp"] >= GATE["minimum_accuracy_gain_pp"]
    and GATE["balanced_accuracy_nondecrease"]
    and GATE["f1_nondecrease"]
    and GATE["auprc_nondecrease"]
    and GATE["sensitivity_loss_no_worse_than_2pp"]
)

comparison = pd.DataFrame([
    {"method":"strict_bridge", **BRIDGE_DEV},
    {"method":"fixed_equal_logit_bridge_qt_stage06", **FIXED_DEV},
])

atomic_csv(
    OUT / "FIXED_EQUAL_LOGIT_DEVELOPMENT_COMPARISON.csv",
    comparison
)

dev_decision = {
    "status":"FIXED_EQUAL_LOGIT_DEVELOPMENT_REPRODUCTION_COMPLETE",
    "branches":ENSEMBLE_BRANCHES,
    "weights":ENSEMBLE_WEIGHTS.tolist(),
    "blend":"equal mean of branch HMM posterior logits",
    "bridge_reference":BRIDGE_DEV,
    "fixed_ensemble":FIXED_DEV,
    "delta_fixed_minus_bridge":delta,
    "gate":GATE,
    "official_x_labels_used":False,
    "candidate_search_performed":False,
    "weight_search_performed":False,
}

atomic_json(
    OUT / "FIXED_EQUAL_LOGIT_DEVELOPMENT_DECISION.json",
    dev_decision
)

display(comparison)

print("\nDELTA FIXED ENSEMBLE - BRIDGE")
for k,v in delta.items():
    print(f"{k:20s}: {v:+.6f} ({v*100:+.3f} pp)")

print("\nGATE")
print(json.dumps(GATE, indent=2))

if not GATE["pass"]:
    raise RuntimeError(
        "Fixed equal-logit ensemble failed development gate. "
        "STOP and retain Strict Bridge."
    )

print("\nDEVELOPMENT GATE: PASS")
print("Proceeding to label-blind final x prediction freeze.")


,method,n,accuracy,balanced_accuracy,precision,sensitivity,specificity,f1,mcc,auroc,auprc,tn,fp,fn,tp
0,strict_bridge,17023,0.880338,0.868884,0.860458,0.820151,0.917618,0.839821,0.74495,0.934627,0.897618,9646,866,1171,5340
1,fixed_equal_logit_bridge_qt_stage06,17023,0.888151,0.874597,0.881943,0.816925,0.932268,0.848190,0.76125,0.941538,0.921381,9800,712,1192,5319



DELTA FIXED ENSEMBLE - BRIDGE
accuracy            : +0.007813 (+0.781 pp)
balanced_accuracy   : +0.005712 (+0.571 pp)
precision           : +0.021486 (+2.149 pp)
sensitivity         : -0.003225 (-0.323 pp)
specificity         : +0.014650 (+1.465 pp)
f1                  : +0.008369 (+0.837 pp)
mcc                 : +0.016300 (+1.630 pp)
auroc               : +0.006912 (+0.691 pp)
auprc               : +0.023763 (+2.376 pp)

GATE
{
  "accuracy_gain_pp": 0.7812958937907566,
  "minimum_accuracy_gain_pp": 0.25,
  "balanced_accuracy_nondecrease": true,
  "f1_nondecrease": true,
  "auprc_nondecrease": true,
  "sensitivity_loss_no_worse_than_2pp": true,
  "pass": true
}

DEVELOPMENT GATE: PASS
Proceeding to label-blind final x prediction freeze.


In [4]:

# ------------------------------------------------------------
# LOAD FINAL MODELS / REPRESENTATIONS
# ------------------------------------------------------------

# Bridge final model + 8-D features
b = np.load(BRIDGE_QREADY, allow_pickle=False)

UID_LEARN = np.asarray(b["learn_uids"]).astype(str)
UID_X = np.asarray(b["test_uids"]).astype(str)
Y_LEARN = np.asarray(b["y_learn"], dtype=np.int8)

P_BRIDGE_LEARN_RAW = linear_head_probability(
    BRIDGE_PT,
    np.asarray(b["Z8_learn"], dtype=np.float32)
)
P_BRIDGE_X_RAW = linear_head_probability(
    BRIDGE_PT,
    np.asarray(b["Z8_test"], dtype=np.float32)
)

# Quantum Transformer final model + 64-D QT representations
q = np.load(QFEATURES, allow_pickle=False)

if not np.array_equal(np.asarray(q["learn_uids"]).astype(str), UID_LEARN):
    raise RuntimeError("QT/Bridge learning UID mismatch")
if not np.array_equal(np.asarray(q["test_uids"]).astype(str), UID_X):
    raise RuntimeError("QT/Bridge x UID mismatch")
if not np.array_equal(np.asarray(q["y_learn"], dtype=np.int8), Y_LEARN):
    raise RuntimeError("QT/Bridge learning-label mismatch")

P_QT_LEARN_RAW = linear_head_probability(
    QT_PT,
    np.asarray(q["qt_angle64_learn"], dtype=np.float32)
)
P_QT_X_RAW = linear_head_probability(
    QT_PT,
    np.asarray(q["qt_angle64_test"], dtype=np.float32)
)

# Corrected Stage06 final model raw saved probabilities
s6 = np.load(STAGE06_FINAL, allow_pickle=False)

if not np.array_equal(np.asarray(s6["learn_uids"]).astype(str), UID_LEARN):
    raise RuntimeError("Stage06/Bridge learning UID mismatch")
if not np.array_equal(np.asarray(s6["test_uids"]).astype(str), UID_X):
    raise RuntimeError("Stage06/Bridge x UID mismatch")
if not np.array_equal(np.asarray(s6["y_learn"], dtype=np.int8), Y_LEARN):
    raise RuntimeError("Stage06/Bridge learning-label mismatch")

P_S6_LEARN_RAW = np.asarray(s6["learn_probability"], dtype=np.float64)
P_S6_X_RAW = np.asarray(s6["test_probability"], dtype=np.float64)

for name,p in {
    "bridge_learn":P_BRIDGE_LEARN_RAW,
    "bridge_x":P_BRIDGE_X_RAW,
    "qt_learn":P_QT_LEARN_RAW,
    "qt_x":P_QT_X_RAW,
    "stage06_learn":P_S6_LEARN_RAW,
    "stage06_x":P_S6_X_RAW,
}.items():
    if not np.isfinite(p).all() or np.any((p<0)|(p>1)):
        raise RuntimeError(f"Invalid probability stream: {name}")

if len(UID_X) != 17248:
    raise RuntimeError(f"Expected 17,248 x rows, got {len(UID_X)}")

print("Learning rows:", len(UID_LEARN))
print("Official-x rows:", len(UID_X))
print("All branch UID/order checks: PASS")
print("Official-x labels loaded: NO")


Learning rows: 17023
Official-x rows: 17248
All branch UID/order checks: PASS
Official-x labels loaded: NO


In [5]:

# ------------------------------------------------------------
# FINAL LEARNING-ONLY TEMPERATURE + HMM FIT
# ------------------------------------------------------------

T_BRIDGE, NLL_BRIDGE = fit_temperature(P_BRIDGE_LEARN_RAW, Y_LEARN)
T_QT, NLL_QT = fit_temperature(P_QT_LEARN_RAW, Y_LEARN)
T_S6, NLL_S6 = fit_temperature(P_S6_LEARN_RAW, Y_LEARN)

CLASS_PRIOR, PI, A = estimate_hmm(
    UID_LEARN,
    Y_LEARN,
)

params = {
    "status":"FINAL_FIXED_EQUAL_LOGIT_PARAMETERS_FROZEN_FROM_LEARNING_ONLY",
    "frozen_at_utc":datetime.now(timezone.utc).isoformat(),
    "branches":ENSEMBLE_BRANCHES,
    "weights":ENSEMBLE_WEIGHTS.tolist(),
    "blend":"equal mean of HMM posterior logits",
    "temperature_bridge":T_BRIDGE,
    "temperature_qt":T_QT,
    "temperature_stage06":T_S6,
    "learning_nll_bridge":NLL_BRIDGE,
    "learning_nll_qt":NLL_QT,
    "learning_nll_stage06":NLL_S6,
    "class_prior":CLASS_PRIOR.tolist(),
    "initial_state_prior":PI.tolist(),
    "transition_matrix":A.tolist(),
    "hmm_lambda":HMM_LAMBDA,
    "hard_threshold":HARD_THRESHOLD,
    "official_x_labels_used":False,
    "search_after_fixed_rule":False,
}

atomic_json(
    OUT / "FINAL_FIXED_EQUAL_LOGIT_LEARNING_ONLY_PARAMETER_FREEZE.json",
    params
)

print("="*100)
print("FINAL FIXED EQUAL-LOGIT PARAMETERS")
print("="*100)
print("Bridge T :", T_BRIDGE)
print("QT T     :", T_QT)
print("Stage06 T:", T_S6)
print("HMM λ    :", HMM_LAMBDA)
print("Threshold:", HARD_THRESHOLD)
print("Official-x labels used: NO")


FINAL FIXED EQUAL-LOGIT PARAMETERS
Bridge T : 0.49739267195793013
QT T     : 0.6717362725971612
Stage06 T: 0.40378345173980323
HMM λ    : 1.0
Threshold: 0.5
Official-x labels used: NO


In [6]:

# ------------------------------------------------------------
# X PREDICTION — LABEL BLIND
# ------------------------------------------------------------

P_BRIDGE_X_HMM = decode_external(
    P_BRIDGE_X_RAW, UID_X,
    T_BRIDGE, CLASS_PRIOR, PI, A
)

P_QT_X_HMM = decode_external(
    P_QT_X_RAW, UID_X,
    T_QT, CLASS_PRIOR, PI, A
)

P_S6_X_HMM = decode_external(
    P_S6_X_RAW, UID_X,
    T_S6, CLASS_PRIOR, PI, A
)

P_ENSEMBLE_X = equal_logit_mean(
    P_BRIDGE_X_HMM,
    P_QT_X_HMM,
    P_S6_X_HMM,
)

YHAT_ENSEMBLE_X = (
    P_ENSEMBLE_X >= HARD_THRESHOLD
).astype(np.int8)

FROZEN_X = (
    OUT / "FINAL_FIXED_EQUAL_LOGIT_X_PREDICTIONS_FROZEN_BEFORE_SCORING.npz"
)

np.savez_compressed(
    FROZEN_X,
    test_uids=UID_X.astype("U128"),
    branch_order=np.asarray(ENSEMBLE_BRANCHES, dtype="U64"),
    weights=ENSEMBLE_WEIGHTS.astype(np.float64),
    blend=np.asarray("equal_logit_mean"),
    bridge_raw_probability=P_BRIDGE_X_RAW.astype(np.float64),
    qt_raw_probability=P_QT_X_RAW.astype(np.float64),
    stage06_raw_probability=P_S6_X_RAW.astype(np.float64),
    bridge_hmm_posterior=P_BRIDGE_X_HMM.astype(np.float64),
    qt_hmm_posterior=P_QT_X_HMM.astype(np.float64),
    stage06_hmm_posterior=P_S6_X_HMM.astype(np.float64),
    ensemble_hmm_logit_mean=P_ENSEMBLE_X.astype(np.float64),
    prediction=YHAT_ENSEMBLE_X.astype(np.int8),
    temperature_bridge=np.asarray(T_BRIDGE, dtype=np.float64),
    temperature_qt=np.asarray(T_QT, dtype=np.float64),
    temperature_stage06=np.asarray(T_S6, dtype=np.float64),
    class_prior=CLASS_PRIOR.astype(np.float64),
    initial_state_prior=PI.astype(np.float64),
    transition_matrix=A.astype(np.float64),
    hmm_lambda=np.asarray(HMM_LAMBDA, dtype=np.float64),
    hard_threshold=np.asarray(HARD_THRESHOLD, dtype=np.float64),
)

freeze = {
    "status":"FINAL_FIXED_EQUAL_LOGIT_X_PREDICTIONS_FROZEN_BEFORE_SCORING",
    "frozen_at_utc":datetime.now(timezone.utc).isoformat(),
    "branches":ENSEMBLE_BRANCHES,
    "weights":ENSEMBLE_WEIGHTS.tolist(),
    "blend":"equal mean of branch HMM posterior logits",
    "development_gate":GATE,
    "prediction_path":str(FROZEN_X),
    "prediction_sha256":sha256_file(FROZEN_X),
    "official_x_labels_read_in_this_notebook":False,
    "official_x_labels_used_for_selection":False,
    "scientific_chronology":(
        "The exact fixed ensemble was proposed after historical official-x results "
        "had already been observed. Subsequent x scoring is post-hoc extended-pipeline "
        "evidence and not a pristine untouched holdout."
    ),
}

atomic_json(
    OUT / "FINAL_FIXED_EQUAL_LOGIT_PRE_SCORE_FREEZE_MANIFEST.json",
    freeze
)

print("="*110)
print("FINAL FIXED EQUAL-LOGIT X PREDICTIONS FROZEN")
print("="*110)
print("Rows:", len(UID_X))
print("Branches:", ENSEMBLE_BRANCHES)
print("Weights:", ENSEMBLE_WEIGHTS.tolist())
print("Prediction SHA256:", freeze["prediction_sha256"])
print("Official-x labels read: NO")
print("\nSTOP. Run the separate scorer only after this freeze exists.")


FINAL FIXED EQUAL-LOGIT X PREDICTIONS FROZEN
Rows: 17248
Branches: ['bridge', 'qt_angle_rx', 'current_stage06']
Weights: [0.3333333333333333, 0.3333333333333333, 0.3333333333333333]
Prediction SHA256: f121a79191be00a28f33e06e7dec20cc689268b10a988c52e21284c90d1e2eef
Official-x labels read: NO

STOP. Run the separate scorer only after this freeze exists.



## Hard stop

Do not modify the ensemble after this point.

The only permitted next operation is the separate scorer, which opens the official labels **after** verifying this prediction artifact's SHA-256.
